In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models
import time

In [ ]:
class FrequencyDataset(Dataset):
    def __init__(self, root_dir):
        self.data_paths = []
        self.labels = []
        
        # 'fake' 폴더는 라벨 0, 'real' 폴더는 라벨 1로 지정
        for label_name, label_idx in [("fake", 0), ("real", 1)]:
            folder_path = os.path.join(root_dir, label_name)
            if os.path.exists(folder_path):
                for file in os.listdir(folder_path):
                    if file.endswith('.npy'):
                        self.data_paths.append(os.path.join(folder_path, file))
                        self.labels.append(label_idx)
                        
    def __len__(self):
        return len(self.data_paths)
        
    def __getitem__(self, idx):
        # 1) .npy 파일 불러오기 -> 크기: (128, 128)
        npy_path = self.data_paths[idx]
        freq_data = np.load(npy_path)
        if freq_data.shape != (128, 128):
            freq_data = np.resize(freq_data, (128, 128)) 
        
        # 2) 파이토치 텐서로 변환
        tensor_data = torch.from_numpy(freq_data).float()
        
        # 3) AI 모델은 채널 차원을 요구하므로 앞에 (1)을 추가 -> 최종 크기: (1, 128, 128)
        tensor_data = tensor_data.unsqueeze(0)
        tensor_data = (tensor_data - tensor_data.mean()) / (tensor_data.std() + 1e-9)
        label = self.labels[idx]
        
        return tensor_data, label

# ==========================================
# 2. 데이터 로드 및 쪼개기 (Train 8 : Test 2)
# ==========================================
frequency_dir = "2_Processed_Data/frequency"
full_dataset = FrequencyDataset(root_dir=frequency_dir)

# 만약 데이터가 너무 적어서 에러가 난다면, 전처리 파이프라인으로 데이터를 더 뽑아오셔야 합니다!
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
# 개선
train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [3]:
# ==========================================
# 3. AI 모델 세팅 (ResNet18 - 1채널 개조 버전)
# ==========================================
# 주파수 데이터는 복잡한 컬러 이미지가 아니므로, 가벼운 ResNet18을 사용합니다.
model = models.resnet18(weights='IMAGENET1K_V1')

# [핵심] 기존 모델은 RGB(3채널)를 받게 되어있음. 이를 주파수(1채널)를 받도록 맨 앞부분 개조!
original_conv1 = model.conv1
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
# (선택 사항) 기존 3채널 가중치를 평균 내서 1채널로 압축해 넣어주면 전이 학습 효과가 조금 더 좋습니다.
model.conv1.weight.data = original_conv1.weight.data.mean(dim=1, keepdim=True)

# 마지막 출력층을 2갈래(Real/Fake)로 개조
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.load_state_dict(torch.load('best_frequency_model.pth'))

<All keys matched successfully>

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 10
best_acc =  0.9783 

start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    model.train()  
    running_loss = 0.0
    corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() 
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1) 
        loss = criterion(outputs, labels)

        loss.backward()  
        optimizer.step() 

        running_loss += loss.item() * inputs.size(0)
        corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = corrects.double() / len(train_dataset)
    print(f'[Train] Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

    model.eval()   
    test_loss = 0.0
    test_corrects = 0

    with torch.no_grad(): 
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            test_corrects += torch.sum(preds == labels.data)

    test_epoch_loss = test_loss / len(test_dataset)
    test_epoch_acc = test_corrects.double() / len(test_dataset)
    print(f'[Test]  Loss: {test_epoch_loss:.4f} | Acc: {test_epoch_acc:.4f}')

    if test_epoch_acc > best_acc:
        best_acc = test_epoch_acc
        torch.save(model.state_dict(), 'best_frequency_model.pth')
        print(" 주파수 모델 최고 성능 갱신! 저장 완료!")
    print()

time_elapsed = time.time() - start_time
print(f' 학습 완료! 총 소요 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초')
print(f' 최고 테스트 정답률: {best_acc:.4f}')

Epoch 1/10
---------------
[Train] Loss: 0.1036 | Acc: 0.9658
[Test]  Loss: 0.1319 | Acc: 0.9466

Epoch 2/10
---------------
[Train] Loss: 0.0644 | Acc: 0.9788
[Test]  Loss: 0.0689 | Acc: 0.9772

Epoch 3/10
---------------
[Train] Loss: 0.0522 | Acc: 0.9834
[Test]  Loss: 0.0637 | Acc: 0.9808
 주파수 모델 최고 성능 갱신! 저장 완료!

Epoch 4/10
---------------


KeyboardInterrupt: 

 0.9783